# CIS 6211 – Foundations of Data Science
## Lab 4: Mathematical Models

**Student Name:** Rana Sultan Alhinidy  
**Course:** CIS 6211 | King Khalid University


In [ ]:
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve
%matplotlib inline

### 📌 Cell Explanation
This cell imports the required libraries:
- **scipy.stats** provides statistical functions used for correlation calculations.
- **pandas** is used for loading the height/weight Excel dataset.
- **matplotlib** is used to draw and visualize distributions and the ROC curve.
- **numpy** is used for array operations and linear interpolation.
- **roc_curve from sklearn** computes the Receiver Operating Characteristic curve for classifier evaluation.

## Cell 1: Height Distribution with Classification Regions
**Data Source:** https://www.statcrunch.com/app/index.php?dataid=1406047

In [ ]:
path = "/content/weight.xls"
data = pd.ExcelFile(path).parse('Feuil1')
men = data[data["Gender"]==1]
women = data[data["Gender"]==0]

bsize = 75
n, bins, patches = plt.hist(men["Height"], bins=range(140,max(men["Height"]),1), density=True, color="blue", alpha=0.4, linewidth=0)
plt.hist(women["Height"], bins=range(140,max(men["Height"]),1), density=True, color="red", alpha=0.4, linewidth=0)

plt.ylim(0,0.065)
plt.vlines(168, 0, 0.08)

plt.text(158, 0.027, "TN", fontsize=16, fontweight="bold")
plt.text(162, 0.007, "FN", fontsize=16, fontweight="bold")
plt.text(169, 0.007, "FP", fontsize=16, fontweight="bold")
plt.text(175, 0.027, "TP", fontsize=16, fontweight="bold")

### 📌 Cell Explanation
This cell loads height and weight data from an Excel file and separates it by gender (1 = male, 0 = female).

Two overlapping histograms are plotted:
- **Blue** = men's height distribution
- **Red** = women's height distribution
- Both are normalized with `density=True` to show probability density rather than raw counts.

A vertical line at **168 cm** serves as the classification decision boundary. The plot is annotated with the four regions of the confusion matrix:
- **TP (True Positive):** Men correctly classified as men (right of 168 in blue region)
- **TN (True Negative):** Women correctly classified as women (left of 168 in red region)
- **FP (False Positive):** Women incorrectly classified as men (right of 168 in red region)
- **FN (False Negative):** Men incorrectly classified as women (left of 168 in blue region)

This visualization shows how overlapping distributions create classification errors, and how moving the threshold changes the tradeoff between error types.

## Cell 2: Unnormalized Histograms (for ROC Curve)

In [ ]:
nM, binM, patchesM = plt.hist(men["Height"], bins=range(140,max(men["Height"]),1), color="blue", alpha=0.4, linewidth=0)
nW, binW, patchesW = plt.hist(women["Height"], bins=range(140,max(men["Height"]),1), color="red", alpha=0.4, linewidth=0)

### 📌 Cell Explanation
This cell plots the same height histograms but **without normalization** (no `density=True`).

The raw counts are stored in:
- `nM` — the count of men in each height bin
- `nW` — the count of women in each height bin
- `binM` — the bin edges

These raw count values are needed in the next cell to **manually calculate the True Positive Rate (TPR) and False Positive Rate (FPR)** at each possible threshold for constructing the ROC curve.

## Cell 3: ROC Curve (Built from Scratch)

In [ ]:
tpr = []
fpr = []
totalW = len(women["Height"])
totalM = len(men["Height"])
sumW = 0
sumM = 0
total = len(data["Height"])

for i in range(len(binM)-1):
    tprate = (totalM-sumM)/float(totalM)
    fprate = (totalW-sumW)/float(totalW)
    tpr.append(tprate)
    fpr.append(fprate)
    sumM += nM[i]
    sumW += nW[i]

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, 'bo', fpr, tpr, 'b-')
plt.plot(np.linspace(0,1,100), np.linspace(0,1,100), 'r-')

plt.xlim(-0.05, 1)
plt.ylim(0,1.05)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve for Gender by Height")

### 📌 Cell Explanation
This cell **manually constructs an ROC curve** by iterating through every possible height threshold.

At each threshold:
- **TPR (True Positive Rate)** = correctly classified men / total men
- **FPR (False Positive Rate)** = incorrectly classified women / total women

As the threshold moves from left (low) to right (high), both TPR and FPR decrease because fewer people are classified as male.

The plot shows:
- **Blue curve** = our ROC curve
- **Red diagonal line** = a random classifier (AUC = 0.5, no better than guessing)

The further the blue curve is from the red line toward the **top-left corner**, the better the classifier. AUC = 1.0 means perfect classification; AUC = 0.5 means no better than random.